In [1]:
import matplotlib.pyplot as plt
import numpy as np
from tapas_gmm_modified.policy.models.tpgmm import (
    ReconstructionStrategy, 
    ModelType,
    FittingStage,
    InitStrategy,
    TPGMMConfig,
    AutoTPGMMConfig,
    AutoTPGMM,
    FrameSelectionConfig,
    DemoSegmentationConfig,
    CascadeConfig,
)

from tapas_gmm_modified.dataset.demos import Demos
from tapas_gmm_modified.viz.gmm import plot_hmm_transition_matrix

np.set_printoptions(precision=2)

plt.style.use('default')

from matplotlib import rc
rc('animation', html='jshtml')

2026-06-10 00:33:45.418 | INFO     |  Running on cpu


/home/jangruhnert/miniconda3/envs/hecarim/lib/python3.10/site-packages/riepybdlib/data.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_listdir


In [2]:
from pathlib import Path

import h5py
from heca.environment.scenes.scene import Scene
from heca.environment.scenes.ogbench.scene import OGBenchScene

from heca.agents.experts.expert import ExpertAgent
from heca.agents.experts.tapas import TapasAgent
scene_cfg = OGBenchScene.Config()
agent_cfg = TapasAgent.Config(
        folder="press_left_button",
        scene=scene_cfg,
    )
scene = OGBenchScene.load(scene_cfg, skip_loading=True)   
agent_dir = ExpertAgent.resolve(agent_cfg)
print(agent_dir)
file_path = Path.cwd().parent.parent / agent_dir / "demos.h5"
save_path = Path.cwd().parent.parent / agent_dir / "policy.pt"
print(file_path)
print(save_path)

2026-06-10 00:33:46.491 | INFO     | heca.misc.logger:info:30 - Running on cpu


/home/jangruhnert/miniconda3/envs/hecarim/lib/python3.10/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/jangruhnert/miniconda3/envs/hecarim/lib/python3.10/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


data/experts/tapas/press_left_button
/home/jangruhnert/Documents/GitHub/master-project/data/experts/tapas/press_left_button/demos.h5
/home/jangruhnert/Documents/GitHub/master-project/data/experts/tapas/press_left_button/policy.pt


In [3]:
from tensordict import TensorDict
import torch
from tapas_gmm_modified.utils.observation import (
    SceneObservation,
    dict_to_tensordict,
)
from heca.entities.entity import Mobility
from heca.misc.td import TDScene, empty_bs

def tapas_td(td_obs: TDScene, td_goal: TDScene, scene: Scene) -> TensorDict:
    action = td_obs.extras["action"]
    reward = td_obs.extras["reward"]
    joint_pos = td_obs["joint_pos"]
    joint_vel = td_obs["joint_vel"]
    cursor = td_obs["cursor"]
    cursor_pos = cursor.position
    cursor_rot = cursor.rotation
    cursor_state = cursor.state
    ee_pose = torch.cat((cursor_pos, cursor_rot), dim=-1)
    ee_state = cursor_state

    # camera_obs = self.image_tensors(obs)

    # multicam_obs = dict_to_tensordict(
    #    {"_order": CameraOrder._create(obs.camera_names)} | camera_obs  # type: ignore
    # )
    poses = {
            entity.cfg.label: torch.cat(
                [
                    td_obs[entity.cfg.label].position,
                    td_obs[entity.cfg.label].rotation,
                ],
                dim=-1,
            )
            for entity in scene.entities
        }
    
    states = {
        entity.cfg.label: td_obs[entity.cfg.label].state
        for entity in scene.entities
    }

    for entity in scene.entities:
        # This adds target frames for mobile entities.
        # Later can be used to set target for the tapas model
        if entity.cfg.mobility == Mobility.FREE:
            pos = td_goal[entity.cfg.label].position
            rot = td_goal[entity.cfg.label].rotation
            state = td_goal[entity.cfg.label].state
            pose = torch.cat((pos, rot), dim=-1)
            poses[f"{entity.cfg.label}_target"] = pose
            states[f"{entity.cfg.label}_target"] = state
    object_poses = dict_to_tensordict(poses)
    object_states = dict_to_tensordict(states)

    return SceneObservation(
            feedback=reward,
            action=action,
            cameras=None,  # multicam_obs,
            ee_pose=ee_pose,
            gripper_state=ee_state,
            object_poses=object_poses,
            object_states=object_states,
            joint_pos=joint_pos,
            joint_vel=joint_vel,
            batch_size=empty_bs,
        )

In [4]:
demos_file = h5py.File(file_path, "r")
# load observations here
print(demos_file.keys())
observations: list[SceneObservation] = []  # type: ignore

demos_scenes, demos_images = scene.load_dataset(demos_file) 
for i, (demo_scenes, demo_images) in enumerate(zip(demos_scenes, demos_images)):
    if not agent_cfg.use_gt:
        raise NotImplementedError("TODO: implement tapas_td and convert demos to tapas format")
        # TODO:
    else:
        obss: list[TensorDict] = []
        for td_scene in demo_scenes:
            td_obs = td_scene
            td_goal = demo_scenes[-1]
            obs = tapas_td(td_obs, td_goal, scene)
            obss.append(obs)
        stacked_obs = TensorDict.stack(obss, dim=0)
        observations.append(stacked_obs)

<KeysViewHDF5 ['actions', 'button_states', 'control', 'demo', 'depth', 'extrinsics', 'intrinsics', 'mask', 'prev_button_states', 'prev_qpos', 'prev_qvel', 'privileged_block_0_pos', 'privileged_block_0_quat', 'privileged_block_0_yaw', 'privileged_button_0_pos', 'privileged_button_0_quat', 'privileged_button_0_vel', 'privileged_button_1_pos', 'privileged_button_1_quat', 'privileged_button_1_vel', 'privileged_drawer_handle_pos', 'privileged_drawer_handle_quat', 'privileged_drawer_handle_yaw', 'privileged_drawer_pos', 'privileged_drawer_vel', 'privileged_target_block', 'privileged_target_block_pos', 'privileged_target_block_quat', 'privileged_target_block_yaw', 'privileged_target_button', 'privileged_target_button_quat', 'privileged_target_button_state', 'privileged_target_button_top_pos', 'privileged_target_drawer_handle_pos', 'privileged_target_drawer_pos', 'privileged_target_window_handle_pos', 'privileged_target_window_pos', 'privileged_window_handle_pos', 'privileged_window_handle_qua

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

In [ ]:
data_kwargs = dict(
    add_init_ee_pose_as_frame=True,
    add_world_frame=False,
    frames_from_keypoints=False,
    kp_indeces=None,
    enforce_z_up=False,
    modulo_object_z_rotation=False,
    make_quats_continuous=True,
)

demos = Demos(observations, **data_kwargs) #type: ignore
print(demos)

In [ ]:
tpgmm_config = TPGMMConfig(
    n_components=20,
    model_type=ModelType.HMM,
    use_riemann=True,
    add_time_component=True,
    add_action_component=False,
    position_only=False,
    add_gripper_action=True,
    reg_shrink=1e-2,
    reg_diag=2e-4,
    reg_diag_gripper=2e-2,
    reg_em_finish_shrink=1e-2,
    reg_em_finish_diag=2e-4,
    reg_em_finish_diag_gripper=2e-2,
    trans_cov_mask_t_pos_corr=False,
    em_steps=50,
    fix_first_component=False, # True maybe
    fix_last_component=False, #True maybe
    reg_init_diag=5e-4,  # 5
    heal_time_variance=False,
)

frame_selection_config = FrameSelectionConfig(
    init_strategy=InitStrategy.TIME_BASED,
    fitting_actions=(FittingStage.INIT,),
    rel_score_threshold=0.0,
    use_bic=False,
    drop_redundant_frames=False,
    gt_frames= [[], [], []] #Frames per segment
)

demos_segmentation_config = DemoSegmentationConfig(
    gripper_based=False,
    distance_based=False,
    velocity_based=True,
    repeat_final_step=0, #1
    repeat_first_step=0,
    components_prop_to_len=True,
    min_n_components=1,
    #velocity_threshold=0.001,
    min_end_distance=20,
)

cascade_config = CascadeConfig(
    kl_keep_time_dim=True,
    kl_keep_rotation_dim=False,
)

auto_tpgmm_config = AutoTPGMMConfig(
    tpgmm=tpgmm_config,
    frame_selection=frame_selection_config,
    demos_segmentation=demos_segmentation_config,
    cascade=cascade_config,
)

In [ ]:
atpgmm = AutoTPGMM(auto_tpgmm_config)


In [ ]:
atpgmm.fit_trajectories(demos, fix_frames=True,
                       init_strategy=InitStrategy.TIME_BASED,
                       fitting_actions=(FittingStage.INIT,)) # FittingStage.EM_HMM))


In [ ]:
atpgmm.plot_model(
    scatter=True, rotations_raw=True, plot_traj=True, plot_gaussians=True,
    annotate_gaussians=False, annotate_trajs=True,
    mean_as_base=False, per_segment=False, gaussian_mean_only=False, plot_traj_means=False) #, size=(150, 10))


In [ ]:
atpgmm.fit_trajectories(demos, fix_frames=True,
                       fitting_actions=(FittingStage.EM_HMM, ))


In [ ]:

atpgmm.plot_model(
    scatter=True, rotations_raw=True, plot_traj=True, plot_gaussians=True,
    annotate_gaussians=False, annotate_trajs=False,
    mean_as_base=False, per_segment=False, gaussian_mean_only=False, plot_traj_means=False)

In [ ]:
atpgmm.plot_model(
    scatter=True, rotations_raw=True, plot_traj=True, plot_gaussians=False,
    annotate_gaussians=True, annotate_trajs=False,
    mean_as_base=False, per_segment=False, gaussian_mean_only=False, plot_traj_means=False, time_based=False)

In [ ]:
atpgmm.plot_hmm_transition_matrix()

In [ ]:
atpgmm.to_disk(save_path) # type: ignore

In [ ]:
seg_local_marginals, seg_trans_marginals, seg_trans_marg_container, seg_joint_models, cascaded_hmms, (reconstructions, original_trajectories, extras) = atpgmm.reconstruct(
    strategy=ReconstructionStrategy.GMR,
    use_ss=False)


In [ ]:
for cascaded_hmm in cascaded_hmms:
    plot_hmm_transition_matrix(cascaded_hmm)

In [ ]:

atpgmm.plot_reconstructions(
    seg_trans_marg_container, cascaded_hmms, reconstructions, original_trajectories,
    plot_trajectories=True, plot_reconstructions=True, plot_gaussians=True,
    time_based=True, equal_aspect=False, per_segment=False)


In [ ]:
atpgmm.plot_reconstructions(
    seg_trans_marginals, seg_joint_models, reconstructions, original_trajectories,
    plot_trajectories=True, plot_reconstructions=True, plot_gaussians=True,
    time_based=False, equal_aspect=True, per_segment=False)
